In [15]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download('stopwords')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
import re
import pandas as pd

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/nayanikap/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/nayanikap/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [16]:
data = pd.read_csv('tripadvisor_hotel_reviews.csv')
#to ensure the data is in lowercase only
data['Review_lowercase'] = data['Review'].str.lower()

In [17]:
en_stopwords = stopwords.words('english')
#removing the word 'not' from the stop words
en_stopwords.remove('not')

In [18]:
#creating a new column of reviews removing the stopwords
data['Review_no_stopwords'] = data['Review_lowercase'].apply(lambda x:' '. join([word for word in x.split() if word not in(en_stopwords)]))
#modifying '*' to 'star' as it has a significant meaning in review, and not be considered as a punctuation
data['Review_no_stopwords_no_punct'] = data.apply(lambda x:re.sub(r"[*]", 'star', x['Review_no_stopwords']), axis=1)
#removing the punctuations from the review
data['Review_no_stopwords_no_punct'] = data.apply(lambda x: re.sub(r"([^\w\s])", "", x['Review_no_stopwords_no_punct']), axis=1)


In [19]:
#tokenizing text
data['tokenized'] = data.apply(lambda x: \
                               word_tokenize( \
                               x['Review_no_stopwords_no_punct'] \
                               ), axis=1)

In [36]:
#stemming the tokens
ps = PorterStemmer()
data['stemmed'] = data['tokenized'].apply(lambda tokens:[ps.stem(token) for token in tokens])
data.head()
lm = WordNetLemmatizer()
data['lemmatized'] = data['stemmed'].apply(lambda tokens: [lm.lemmatize(token) for token in tokens])
data.head()
#finding the most commonly occurring words in data set
tokens_clean = sum(data['lemmatized'], [])
unigrams = (pd.Series(nltk.ngrams(tokens_clean,1)).value_counts())
print(unigrams)
bigrams = (pd.Series(nltk.ngrams(tokens_clean,2)).value_counts())
print(bigrams)

(hotel,)           292
(room,)            275
(stay,)            160
(great,)           126
(not,)             122
                  ... 
(australiaour,)      1
(australian,)        1
(singaporew,)        1
(brilliant,)         1
(staffaft,)          1
Name: count, Length: 2274, dtype: int64
(great, locat)     24
(space, needl)     21
(hotel, monaco)    16
(stay, hotel)      15
(pike, place)      12
                   ..
(like, intrud)      1
(guest, like)       1
(like, guest)       1
(didnt, make)       1
(room, offer)       1
Name: count, Length: 8130, dtype: int64
